# FLAG 2027 - Notebook 12: New encoders (stage 1: challenge + MAV-Celeb)

Notebook 11 showed the head is saturated -> the information in the FEATURES is the limit. Candidates from our literature search:

| Name | Modality | Model | Why |
|---|---|---|---|
| `agv` | face | `abhilash88/age-gender-prediction` (ViT) | explicit AGE + gender cues; used by the FAME 2026 winner |
| `siglip2` | face | `google/siglip2-so400m-patch14-384` | newest strong image-text model (successor of CLIP) |
| `r293` | voice | WeSpeaker ResNet293-LM (ONNX) | very strong speaker model, VoxCeleb-only training (Bengali-safe) |

Extracted for challenge train + dev and MAV-Celeb v1+v3 (VoxCeleb raw files were deleted; stage 2 only if this helps). Then feature combinations are compared in the **challenge + MAV-Celeb** setting (reference E2_mav_dn = 27.84).

Every encoder is wrapped in a safety net: if one fails to load, the others continue. Resumable.

**How to run:** T4 GPU, Run all. First run about 2-3 hours (unpacking + extraction + training). Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive, GPU, packages (GPU onnxruntime installed LAST so it is not replaced by the CPU build)
from google.colab import drive
drive.mount('/content/drive')
!pip -q uninstall -y onnxruntime onnxruntime-gpu
!pip -q install onnxruntime-gpu huggingface_hub onnx onnx2torch
import onnxruntime as ort
print('onnxruntime providers:', ort.get_available_providers())
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - paths; unzip challenge (all files) + needed MAV-Celeb files to local disk; build file lists
import os, re, json, time, zipfile, shutil, copy
import numpy as np, pandas as pd
import librosa, soundfile as sf
from PIL import Image
from tqdm.auto import tqdm

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
RESDIR = os.path.join(PROJECT, 'results')
CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    if not os.path.exists(os.path.join(FULL, z + '.done')):
        t0 = time.time()
        shutil.copy(os.path.join(ZIP_DIR, z), '/content/' + z)
        with zipfile.ZipFile('/content/' + z) as zf:
            zf.extractall(FULL)
        os.remove('/content/' + z); open(os.path.join(FULL, z + '.done'), 'w').close()
        print('%s unzipped in %.0f s' % (z, time.time() - t0))
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
    return None

SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
LISTS = {'train': (find_one('train_English.txt'), 2, 3)}
for s in SPLIT_NAMES[1:]:
    track, lang = s.rsplit('_', 1)
    LISTS[s] = (find_one('/' + track + '/', lang + '_test.txt'), 1, 2)
FILES = {}   # split -> (voice paths, face paths), row order = list order = CSV order
for s, (lp, cv, cf) in LISTS.items():
    rows = [l.split() for l in open(lp) if l.strip()]
    FILES[s] = ([resolve(r[cv], lp) for r in rows], [resolve(r[cf], lp) for r in rows])
    assert None not in FILES[s][0] and None not in FILES[s][1], 'missing files in ' + s

# MAV-Celeb v1 + v3: unpack only the files in the train lists, in the SAME order as Notebook 06
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'
mv, mf, mspk = [], [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']),
                          ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.done')
    need = not os.path.exists(flag)
    if need:
        shutil.copy(os.path.join(MAV_ZIPS, zname), '/content/' + zname)
    zf = zipfile.ZipFile(os.path.join('/content', zname) if need else os.path.join(MAV_ZIPS, zname))
    zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            vp, fp = os.path.join(MAVD, ver, vm), os.path.join(MAVD, ver, fm)
            if need:
                zf.extract(vm, os.path.join(MAVD, ver)); zf.extract(fm, os.path.join(MAVD, ver))
            mv.append(vp); mf.append(fp); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        os.remove('/content/' + zname); open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV-Celeb row order differs from Notebook 06!'
FILES['mav'] = (mv, mf)
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
note('Files ready: ' + ', '.join('%s %d' % (k, len(v[0])) for k, v in FILES.items()))

In [ ]:
# Cell 3 - the three new encoders (each returns one embedding per file)
# v2 (17 Oct): age-gender ViT processor fix (crop_size bug) + ResNet293 run in PyTorch on GPU (onnxruntime's cuDNN failed)
from transformers import AutoModel, AutoImageProcessor, AutoProcessor

def as_tensor(o):
    if torch.is_tensor(o):
        return o
    for a in ['image_embeds', 'pooler_output']:
        if getattr(o, a, None) is not None:
            return getattr(o, a)
    return o.last_hidden_state[:, 0]

def enc_siglip2(paths, bs=32):
    name = 'google/siglip2-so400m-patch14-384'
    model = AutoModel.from_pretrained(name, dtype=torch.float16).to(dev).eval()
    proc = AutoProcessor.from_pretrained(name)
    out = []
    for i in tqdm(range(0, len(paths), bs), desc='siglip2'):
        ims = [Image.open(p).convert('RGB') for p in paths[i:i + bs]]
        x = proc(images=ims, return_tensors='pt')['pixel_values'].to(dev, torch.float16)
        with torch.no_grad():
            out.append(as_tensor(model.get_image_features(pixel_values=x)).float().cpu().numpy())
    return np.concatenate(out)

def enc_agv(paths, bs=64):
    # ViT age-gender model (FAME 2026 winner's face attribute encoder): CLS token of its ViT backbone.
    # Its preprocessor_config says do_center_crop=True but gives no crop_size -> we resize straight to 224x224.
    name = 'abhilash88/age-gender-prediction'
    model = AutoModel.from_pretrained(name, trust_remote_code=True).to(dev).eval()
    try:
        proc = AutoImageProcessor.from_pretrained(name, trust_remote_code=True)
    except Exception:
        proc = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
    proc.do_center_crop = False
    proc.size = {'height': 224, 'width': 224}
    backbone = getattr(model, 'vit', None) or getattr(model, 'backbone', None) or model
    out = []
    for i in tqdm(range(0, len(paths), bs), desc='age-gender ViT'):
        ims = [Image.open(p).convert('RGB') for p in paths[i:i + bs]]
        x = proc(images=ims, return_tensors='pt')['pixel_values'].to(dev)
        with torch.no_grad():
            o = backbone(pixel_values=x)
        h = o.last_hidden_state[:, 0] if getattr(o, 'last_hidden_state', None) is not None else as_tensor(o)
        out.append(h.float().cpu().numpy())
    return np.concatenate(out)

def enc_r293(paths, sec=6, bs=32):
    # WeSpeaker ResNet293-LM (VoxCeleb2-trained). The ONNX file is converted to a PyTorch model (onnx2torch)
    # and run on the GPU in batches. Every clip is made exactly `sec` seconds long (centre crop, or repeat if
    # shorter) so all inputs have the same shape -> fast batching and no cuDNN shape problems.
    from huggingface_hub import list_repo_files, hf_hub_download
    import torchaudio
    from onnx2torch import convert
    repo = 'Wespeaker/wespeaker-voxceleb-resnet293-LM'
    onnx_name = [f for f in list_repo_files(repo) if f.endswith('.onnx')][0]
    path = hf_hub_download(repo, onnx_name, local_dir=os.path.join(CACHE, 'wespeaker_r293'))
    net = convert(path).to(dev).eval()
    L = 16000 * sec
    def fbank(p):
        w = librosa.load(p, sr=16000, mono=True)[0]
        if len(w) >= L:
            s0 = (len(w) - L) // 2; w = w[s0:s0 + L]
        else:
            w = np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L]
        wt = torch.from_numpy(w).unsqueeze(0) * (1 << 15)
        fb = torchaudio.compliance.kaldi.fbank(wt, num_mel_bins=80, frame_length=25, frame_shift=10, dither=0.0,
                                               sample_frequency=16000, window_type='hamming', use_energy=False)
        return fb - fb.mean(0, keepdim=True)
    # speed check on the first batch: must be GPU-fast
    out, t0 = [], time.time()
    for i in tqdm(range(0, len(paths), bs), desc='resnet293'):
        x = torch.stack([fbank(p) for p in paths[i:i + bs]]).to(dev)
        with torch.no_grad():
            e = net(x)
        e = e[0] if isinstance(e, (list, tuple)) else e
        out.append(e.reshape(len(x), -1).float().cpu().numpy())
        if i == 0:
            print('first batch: %.1f s for %d clips' % (time.time() - t0, len(x)))
    return np.concatenate(out)

ENCODERS = {'agv': (1, enc_agv), 'siglip2': (1, enc_siglip2), 'r293': (0, enc_r293)}   # 0 = voice files, 1 = face files
print('encoders defined')

In [ ]:
# Cell 4 - extract + save (one .npz per encoder with challenge splits + 'mav'; skipped if it exists)
ORDER = SPLIT_NAMES + ['mav']
for name, (kind, fn) in ENCODERS.items():
    out_path = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(out_path):
        print(name, 'already extracted'); continue
    t0 = time.time()
    try:
        allp = [p for s in ORDER for p in FILES[s][kind]]
        emb = fn(allp)
    except Exception as e:
        note('ENCODER %s FAILED: %r' % (name, e)); continue
    saved, i = {}, 0
    for s in ORDER:
        n = len(FILES[s][kind]); saved[s] = emb[i:i + n].astype(np.float32); i += n
    np.savez(out_path, **saved)
    note('%-8s dim %4d | %d files in %.1f min' % (name, emb.shape[1], len(allp), (time.time() - t0) / 60))

In [ ]:
# Cell 5 - training setup: challenge + MAV-Celeb, per-dataset normalization (E2_mav_dn recipe)
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve

def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['ecapa'] = {'train': Xv, 'mav': MAV['ecapa']}
for s in SPLIT_NAMES[1:]:
    track, lang = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in SPLIT_NAMES}; FEAT[name]['mav'] = MAV[name]
for name in ENCODERS:
    p = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(p):
        d = np.load(p); FEAT[name] = {s: d[s] for s in ORDER}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y)
id_gender = {c: spk2g[cls2spk[c]] for c in ids}
N_CH, N_MAV = len(y), len(MAV['spk'])
mav_lab = {s: 1000 + i for i, s in enumerate(sorted(set(MAV['spk'])))}
Y_ALL = np.r_[y, np.array([mav_lab[s] for s in MAV['spk']])]
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV, int)]
EXT = N_CH + np.arange(N_MAV)

def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav')])

def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))

class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused)
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf_, sf_, mv_, sv_ = model.norm
    ft, vt, _, _ = model.embed((xf - mf_) / sf_, (xv - mv_) / sv_)
    return -(ft - vt).norm(dim=1).cpu().numpy()   # HIGHER = same person
print('training setup ready | features available:', list(FEAT))

In [ ]:
# Cell 6 - compare feature combinations (challenge + MAV-Celeb, per-dataset normalization) - RESUMABLE
SEEDS = [1, 2, 3]
P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60

def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch

def train_combo(F_raw, V_raw, tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = dnorm(F_raw, V_raw, all_tr)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev)
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev)
    model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)
    mf_, sf_, mv_, sv_ = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf_) / sf_, (XV[tr[j]] - mv_) / sv_)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch

COMBOS = {
    'N0_clip|ecapa+wavlm':          (['clip'], ['ecapa', 'wavlm_sv']),
    'N1_clip+agv|ecapa+wavlm':      (['clip', 'agv'], ['ecapa', 'wavlm_sv']),
    'N2_siglip2|ecapa+wavlm':       (['siglip2'], ['ecapa', 'wavlm_sv']),
    'N3_siglip2+agv|ecapa+wavlm':   (['siglip2', 'agv'], ['ecapa', 'wavlm_sv']),
    'N4_clip|ecapa+wavlm+r293':     (['clip'], ['ecapa', 'wavlm_sv', 'r293']),
    'N5_clip+siglip2+agv|all3':     (['clip', 'siglip2', 'agv'], ['ecapa', 'wavlm_sv', 'r293']),
}
COMBOS = {k: v for k, v in COMBOS.items() if all(n in FEAT for n in v[0] + v[1])}
print('Combos that will run:', list(COMBOS))
CSV = os.path.join(RESDIR, 'nb12_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['combo', 'seed', 'fold'])
t0 = time.time()
for name, (fn_, vn_) in COMBOS.items():
    F_raw, V_raw = build_train(fn_), build_train(vn_)
    for seed in SEEDS:
        for k in range(K):
            if ((done.combo == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep, XF, XV, _ = train_combo(F_raw, V_raw, tr_idx, vsets, seed=seed)
            e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
            row = pd.DataFrame([dict(combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-28s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min'
                  % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))

In [ ]:
# Cell 7 - results + summary (stage 2 = re-extract VoxCeleb2 with the winning encoders, only if clearly better)
R = pd.read_csv(CSV)
R['overall'] = (R['std'] + R['gender']) / 2
per_seed = R.groupby(['combo', 'seed'])[['std', 'gender', 'overall']].mean().reset_index()
tab = per_seed.groupby('combo').agg(std=('std', 'mean'), gender=('gender', 'mean'),
                                    overall=('overall', 'mean'), seed_sd=('overall', 'std'))
tab = tab.loc[[c for c in COMBOS if c in tab.index]].round(2)
note('NEW ENCODERS (challenge + MAV-Celeb, per-dataset norm, 5-fold x 3 seeds; reference E2_mav_dn = 27.84)')
note(tab.to_string())
BEST = tab['overall'].idxmin()
note('BEST: %s (%.2f) vs N0 %.2f' % (BEST, tab.loc[BEST, 'overall'], tab.loc['N0_clip|ecapa+wavlm', 'overall']))
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)

In [ ]:
# Cell 8 - final model for the BEST combo (all 70 challenge people + MAV-Celeb, 3 seeds) -> submission ZIP
# also saves the raw dev scores (HIGHER = same) so we can ensemble systems later without retraining
LOWER_IS_SAME = True
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
fn_, vn_ = COMBOS[BEST]
F_raw, V_raw = build_train(fn_), build_train(vn_)
ep = int(np.median(R[R.combo == BEST]['best_ep']))
finals = []
for s in SEEDS:
    m, _, _, _, ch = train_combo(F_raw, V_raw, np.arange(N_CH), None, seed=s, max_epochs=ep)
    finals.append(m)
tag = BEST.split('_')[0]
OUT = '/content/sub_12_' + tag
dev_scores = {}
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang, fname in LANG_FILE.items():
        split = track + '_' + lang
        pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang + '_test.txt')) if l.strip()]
        fx = ((build(fn_, split) - ch[0]) / ch[1]).astype(np.float32)      # challenge TRAIN statistics
        vx = ((build(vn_, split) - ch[2]) / ch[3]).astype(np.float32)
        xf = torch.tensor(fx, device=dev); xv = torch.tensor(vx, device=dev)
        s_ = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
        dev_scores[split] = s_
        out = -s_ if LOWER_IS_SAME else s_
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, out):
                f.write('%s %.6f\n' % (pid, v))
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
ZIP_PATH = os.path.join(SUBDIR, '12_%s_clip_ecapa_wavlm_r293.zip' % tag)
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
np.savez(os.path.join(RESDIR, 'devscores_12_%s.npz' % tag), **dev_scores)
note('Submission ZIP saved: %s (%d epochs) + raw dev scores saved for ensembling' % (ZIP_PATH[len(BASE):], ep))

LOG = os.path.join(PROJECT, 'experiments.csv')
log = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame()
if len(log) and 'run' in log:
    log = log[~log.run.isin(['12_' + c for c in tab.index])]
new = pd.DataFrame([{'date': time.strftime('%Y-%m-%d %H:%M'), 'run': '12_' + c, 'change': 'challenge+MAV dn, face=%s voice=%s' % ('+'.join(COMBOS[c][0]), '+'.join(COMBOS[c][1])),
                     'seed': '1,2,3', 'cv_std': tab.loc[c, 'std'], 'cv_gender': tab.loc[c, 'gender'],
                     'cv_overall': tab.loc[c, 'overall'], 'codabench_dev': ''} for c in tab.index])
pd.concat([log, new], ignore_index=True).to_csv(LOG, index=False)
print('=' * 70)
for s in SUMMARY[-2:]:
    print(s)
print('=' * 70)